# Ch.2 — Collaborative Filtering _(Solution)_

> **The story.** In **1994**, Paul Resnick and colleagues coined the term "collaborative filtering" in the **GroupLens** paper — a system where Usenet readers collaboratively filtered articles by rating them, and those ratings were used to recommend articles to similar readers. The insight was radical: you don't need to understand *what* an article (or movie) is about. You only need to know *who liked it*. If Alice and Bob vote identically on 50 articles, Alice will probably like what Bob rated highly but hasn't yet seen. Seven years later, **Sarwar, Karypis, Konstan and Riedl** (2001) published the paper Amazon had quietly patented in 1998: **item-based collaborative filtering**. Flip the perspective — instead of finding similar *users*, find similar *items*. "Star Wars" and "Blade Runner" attract similar ratings from similar people; recommend one to fans of the other. This proved more scalable: items don't change their preferences, but users do. The Netflix Prize (2006–2009) showed that CF alone could get within 6% of the winning solution. Today, CF is the backbone of Spotify, YouTube, and TikTok recommendations. The core insight, unchanged since 1994: **your taste is encoded in your ratings history, and people with similar history will like similar things in the future**.
>
> **Where you are.** Chapter two of the Recommender Systems track. The popularity baseline from Ch.1 gave every user the same 10 movies — HR@10 ≈ 42%. Now you personalise: find users with similar taste (user-based CF) or find movies with similar rating patterns (item-based CF), and recommend accordingly. This is the first time FlixAI treats different users differently.
>
> **Notation.** $r_{ui}$ — rating by user $u$ on item $i$ (1–5, or missing if unrated); $\text{sim}(u, v)$ — cosine similarity between users $u$ and $v$; $\mathcal{N}_k(u)$ — the $k$ nearest neighbours of user $u$; $\bar{r}_u$ — mean rating of user $u$ across all rated items; $\hat{r}_{ui}$ — predicted rating for user $u$ on item $i$; $I_{uv}$ — set of items co-rated by both $u$ and $v$; $K$ — neighbourhood size hyperparameter.

---

## 0 · The Challenge

> **The mission**: FlixAI — >85% HR@10 across 5 constraints.

**What we know so far:**
- Ch.1: Popularity baseline = **42% HR@10** — everyone gets the same 10 movies
- MovieLens 100k: 943 users, 1,682 movies, 93.7% sparse
- Evaluation framework established (HR@10, NDCG@10, leave-one-out split)
- **But zero personalisation — a 20-year-old action fan and a 60-year-old romance lover see identical recommendations**

**What's blocking us:**
The popularity baseline treats User 1 (who only rates horror films) and User 2 (who only rates romance films) identically. Both see "The Shawshank Redemption" and "Pulp Fiction" at the top of their list. The system doesn't even read the ratings column — it just counts. Your VP of Product: *"If we're just showing everyone the popular movies, why do we need machine learning? We could do this in Excel."*

**What this chapter unlocks:**
- **User-based CF**: Find the K most similar users, weight their ratings → personalised prediction
- **Item-based CF**: Find items similar to what you already rated → stable + scalable
- **Explainability**: "Users who liked Star Wars also liked Blade Runner"
- **HR@10 jumps from 42% → ~68%** — 26 points from personalisation alone

## 1 · The Core Idea

The key observation driving collaborative filtering: you don't need to know anything about what a movie *is* — genre, director, cast. You only need to know who rated it and how highly. If User 12 and User 47 have both rated the same 30 films similarly, they share a taste profile. What User 12 loved that User 47 hasn't seen yet is a strong recommendation signal. Scale this to 943 users and 100,000 ratings and the signal becomes powerful.

Two flavours exist, and they answer different questions:

- **User-Based CF**: "Find users like me — what did they watch that I haven't?" Personalised but expensive at scale: every new rating requires recomputing all pairwise user similarities.
- **Item-Based CF**: "Find movies like the ones I already rated." More stable: item similarity doesn't change when a new user joins. Precompute once, serve fast.

Both rely on cosine similarity applied to the sparse user-item matrix — no content features, no metadata, no genre labels. The prediction formula for user-based CF centers each user's ratings around their personal mean, so a user who always rates 4–5 stars doesn't artificially inflate the prediction for a user who rates 1–3:

$$\hat{r}_{ui} = \bar{r}_u + \frac{\sum_{v \in \mathcal{N}_k(u)} \text{sim}(u, v) \cdot (r_{vi} - \bar{r}_v)}{\sum_{v \in \mathcal{N}_k(u)} |\text{sim}(u, v)|}$$

> **Optional depth:** Pearson correlation is equivalent to mean-centered cosine similarity on the co-rated items. Mean-centering removes user bias (generous vs harsh raters); cosine on co-rated items handles sparsity. See [MathUnderTheHood ch07](../../00-math-under-the-hood/ch07) for the derivation.

```mermaid
graph TD
    A["User A likes\nMovies 1,2,3"] --> C{"Find similar..."}
    B["User B likes\nMovies 1,2,4"] --> C
    C --> D["User-based CF:\nA and B are similar\nrecommend Movie 4 to A"]
    C --> E["Item-based CF:\nMovies 1,2 co-rated\nrecommend Movie 3\nto B's queue"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
np.random.seed(SEED)

print("Libraries loaded.")

In [ ]:
# ── Load MovieLens 100k ───────────────────────────────────────────────────
url = "https://files.grouplens.org/datasets/movielens/ml-100k/"

ratings = pd.read_csv(
    url + "u.data", sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"]
)

n_users = ratings["user_id"].nunique()
n_items = ratings["item_id"].nunique()
print(f"Users: {n_users}  Items: {n_items}  Ratings: {len(ratings):,}")

In [ ]:
# ── Leave-One-Out Split ───────────────────────────────────────────────────
def leave_one_out_split(ratings_df):
    """Hold out the last rating per user (by timestamp) for testing."""
    ratings_sorted = ratings_df.sort_values("timestamp")
    test = ratings_sorted.groupby("user_id").tail(1).copy()
    train = ratings_sorted.drop(test.index).copy()
    return train, test

train, test = leave_one_out_split(ratings)
print(f"Train: {len(train):,}  Test: {len(test):,}")

In [ ]:
# ── Build Sparse User-Item Matrix ─────────────────────────────────────────
def build_sparse_matrix(train_df, n_users, n_items):
    """Create sparse CSR matrix from rating triplets (0-indexed)."""
    row = train_df["user_id"].values - 1
    col = train_df["item_id"].values - 1
    data = train_df["rating"].values.astype(np.float32)
    return csr_matrix((data, (row, col)), shape=(n_users, n_items))

R = build_sparse_matrix(train, n_users, n_items)
print(f"User-item matrix: {R.shape}")
print(f"Non-zero entries: {R.nnz:,}")
print(f"Sparsity: {1 - R.nnz / (R.shape[0] * R.shape[1]):.1%}")

In [ ]:
# ── Evaluation Metrics ────────────────────────────────────────────────────
def hit_rate_at_k(test_df, top_k_per_user, k=10):
    hits = 0
    for _, row in test_df.iterrows():
        user = row["user_id"]
        test_item = row["item_id"]
        recs = top_k_per_user.get(user, [])[:k]
        if test_item in recs:
            hits += 1
    return hits / len(test_df)

def ndcg_at_k(test_df, top_k_per_user, k=10):
    ndcgs = []
    for _, row in test_df.iterrows():
        user = row["user_id"]
        test_item = row["item_id"]
        recs = top_k_per_user.get(user, [])[:k]
        if test_item in recs:
            rank = recs.index(test_item) + 1
            ndcgs.append(1.0 / np.log2(rank + 1))
        else:
            ndcgs.append(0.0)
    return np.mean(ndcgs)

print("Evaluation functions defined.")

## 2 · User-Based Collaborative Filtering

The popularity baseline gave every user the same list. Here you fix that: find the $k$ users whose rating history most resembles the current user's, then use their ratings (weighted by similarity) to predict what the current user would give unrated films. Two users who agreed on 30 films are likely to agree on the 31st — the question is only how much weight to give each neighbour.

The mean-centering step matters. A user who always gives 4–5 stars and a user who always gives 1–2 stars can actually have very similar *taste preferences* — they just calibrate their scale differently. By subtracting each user's mean before computing similarity, you compare deviations from personal baseline rather than raw scores.

Find the $k$ most similar users (cosine on mean-centered ratings) and predict:

$$\hat{r}_{ui} = \bar{r}_u + \frac{\sum_{v \in \mathcal{N}_k(u)} \text{sim}(u, v) \cdot (r_{vi} - \bar{r}_v)}{\sum_{v \in \mathcal{N}_k(u)} |\text{sim}(u, v)|}$$

> **Optional depth:** The choice of $k$ trades bias for variance. Small $k$: personalised but noisy (few neighbours). Large $k$: stable but diluted (distant neighbours pollute the signal). The typical sweet spot for MovieLens 100k is $k = 20$–$50$.

In [ ]:
# ── User-Based CF ─────────────────────────────────────────────────────────
def user_based_cf(R_sparse, k_neighbors=50, n_recs=10):
    """User-based CF using cosine similarity on the sparse matrix."""
    R_dense = R_sparse.toarray()

    # Mean-center each user's ratings (only non-zero entries)
    user_means = np.zeros(R_dense.shape[0])
    R_centered = R_dense.copy()
    for u in range(R_dense.shape[0]):
        rated_mask = R_dense[u] > 0
        if rated_mask.sum() > 0:
            user_means[u] = R_dense[u, rated_mask].mean()
            R_centered[u, rated_mask] -= user_means[u]
            R_centered[u, ~rated_mask] = 0  # keep unrated as 0

    # User-user similarity (cosine on centered ratings)
    user_sim = cosine_similarity(csr_matrix(R_centered))
    np.fill_diagonal(user_sim, 0)

    recommendations = {}
    for u in range(R_dense.shape[0]):
        rated_mask = R_dense[u] > 0
        # Top-k similar users
        neighbor_sims = user_sim[u]
        top_neighbors = np.argsort(neighbor_sims)[-k_neighbors:]

        scores = np.zeros(R_dense.shape[1])
        for i in range(R_dense.shape[1]):
            if rated_mask[i]:
                continue  # skip already rated
            # Neighbors who rated item i
            neighbor_ratings = R_dense[top_neighbors, i]
            valid = neighbor_ratings > 0
            if valid.sum() == 0:
                continue
            sims = neighbor_sims[top_neighbors][valid]
            devs = neighbor_ratings[valid] - user_means[top_neighbors][valid]
            denom = np.abs(sims).sum()
            if denom > 0:
                scores[i] = user_means[u] + (sims * devs).sum() / denom

        top_items = np.argsort(scores)[-n_recs:][::-1]
        recommendations[u + 1] = (top_items + 1).tolist()

    return recommendations

print("Computing user-based CF (this may take a minute)...")
top_k_user_cf = user_based_cf(R, k_neighbors=50, n_recs=10)

hr_ucf = hit_rate_at_k(test, top_k_user_cf, k=10)
ndcg_ucf = ndcg_at_k(test, top_k_user_cf, k=10)
print(f"\nUser-Based CF:")
print(f"  HR@10   = {hr_ucf:.3f} ({hr_ucf*100:.1f}%)")
print(f"  NDCG@10 = {ndcg_ucf:.4f}")

**Reflection — § 2 User-Based CF**

User-based CF achieves ~60% HR@10 — a substantial jump from the 42% popularity baseline. The personalisation is real: two users with opposite tastes now receive entirely different top-10 lists. But two problems emerge immediately:

1. **Sparsity**: With 93.7% of the matrix empty, most user pairs co-rate fewer than 5 movies. A cosine similarity computed on 3 shared films is statistically meaningless, yet the model uses it as if it were reliable.
2. **Scalability**: Computing pairwise user similarity requires an $O(m^2)$ pass. With 943 users that's 888,000 pairs — manageable now, but at 1M users it's 10^12 operations.

Item-based CF addresses both: items have more ratings per entity than users do, and item similarities are precomputable and stable (a movie's genre doesn't change when a new user joins).

## 3 · Item-Based Collaborative Filtering

The sparsity problem in user-CF is a data problem, not an algorithm problem. You can sidestep it by flipping the similarity axis: instead of finding users similar to you, find *items* similar to the items you already rated, then score those similar items using your own ratings. The key advantage: items accumulate ratings from hundreds of users, making item-item similarities more statistically robust than user-user similarities in a sparse matrix.

The trade-off is interpretability: "Blade Runner is similar to Star Wars" is something you can explain to a user. "User 47 is similar to User 12" is not.

Item similarities are precomputed once — they don't change when a new user joins. For user $u$, the predicted score for unrated item $i$ is:

$$\hat{r}_{ui} = \frac{\sum_{j \in \mathcal{N}_k(i)} \text{sim}(i, j) \cdot r_{uj}}{\sum_{j \in \mathcal{N}_k(i)} |\text{sim}(i, j)|}$$

where $\mathcal{N}_k(i)$ is the set of the $k$ items most similar to $i$ that user $u$ has already rated.

> **Optional depth:** The denominator normalises by absolute similarity sum rather than raw sum — this prevents items with one very similar neighbour from dominating over items with many moderately similar neighbours. Without the absolute value, negative similarities (items that anti-correlate) could destructively cancel positive ones.

In [ ]:
# ── Item-Based CF ─────────────────────────────────────────────────────────
def item_based_cf(R_sparse, k_neighbors=30, n_recs=10):
    """Item-based CF using cosine similarity."""
    # Item-item similarity (cosine on columns)
    item_sim = cosine_similarity(R_sparse.T)
    np.fill_diagonal(item_sim, 0)

    R_dense = R_sparse.toarray()
    recommendations = {}

    for u in range(R_dense.shape[0]):
        rated_items = np.where(R_dense[u] > 0)[0]
        user_ratings = R_dense[u]

        scores = np.zeros(R_dense.shape[1])
        for i in range(R_dense.shape[1]):
            if user_ratings[i] > 0:
                continue
            # Similarity to items user has rated
            sims = item_sim[i, rated_items]
            top_k = np.argsort(sims)[-k_neighbors:]
            top_sims = sims[top_k]
            top_ratings = user_ratings[rated_items[top_k]]

            denom = np.abs(top_sims).sum()
            if denom > 0:
                scores[i] = (top_sims * top_ratings).sum() / denom

        top_items = np.argsort(scores)[-n_recs:][::-1]
        recommendations[u + 1] = (top_items + 1).tolist()

    return recommendations

print("Computing item-based CF (this may take a minute)...")
top_k_item_cf = item_based_cf(R, k_neighbors=30, n_recs=10)

hr_icf = hit_rate_at_k(test, top_k_item_cf, k=10)
ndcg_icf = ndcg_at_k(test, top_k_item_cf, k=10)
print(f"\nItem-Based CF:")
print(f"  HR@10   = {hr_icf:.3f} ({hr_icf*100:.1f}%)")
print(f"  NDCG@10 = {ndcg_icf:.4f}")

In [ ]:
# ── Compare Methods ───────────────────────────────────────────────────────
results = pd.DataFrame({
    "Method": ["Popularity (Ch.1)", "User-Based CF", "Item-Based CF"],
    "HR@10": [0.42, hr_ucf, hr_icf],
    "NDCG@10": [0.0, ndcg_ucf, ndcg_icf]
})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ["#95a5a6", "#3498db", "#e67e22"]
axes[0].bar(results["Method"], results["HR@10"] * 100, color=colors, edgecolor="white")
axes[0].axhline(85, color="red", linestyle="--", alpha=0.7, label="Target: 85%")
axes[0].set(ylabel="Hit Rate@10 (%)", title="Hit Rate@10 Comparison")
axes[0].legend()

axes[1].bar(results["Method"], results["NDCG@10"], color=colors, edgecolor="white")
axes[1].set(ylabel="NDCG@10", title="NDCG@10 Comparison")

plt.suptitle("Collaborative Filtering vs Popularity Baseline", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig("img/cf_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(results.to_string(index=False))

In [ ]:
# ── Effect of k (Neighbors) on Item-Based CF ──────────────────────────────
k_values_nb = [5, 10, 20, 30, 50, 100]
hr_by_k = []

for k in k_values_nb:
    print(f"  Computing item-CF with k={k}...")
    recs = item_based_cf(R, k_neighbors=k, n_recs=10)
    hr_by_k.append(hit_rate_at_k(test, recs, k=10))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_values_nb, [h * 100 for h in hr_by_k], "o-", color="#e67e22", linewidth=2, markersize=8)
ax.set(xlabel="k (Number of Neighbors)", ylabel="Hit Rate@10 (%)",
       title="Item-Based CF: Effect of Neighbor Count")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("img/k_neighbors_effect.png", dpi=150, bbox_inches="tight")
plt.show()

for k, hr in zip(k_values_nb, hr_by_k):
    print(f"  k={k:>3d}: HR@10 = {hr:.3f} ({hr*100:.1f}%)")

In [ ]:
# ── Visualise Item Similarity Matrix (Top-20 Movies) ─────────────────────
item_sim_full = cosine_similarity(R.T)
np.fill_diagonal(item_sim_full, 0)

# Top 20 most-rated movies
top_20_ids = ratings.groupby("item_id").size().nlargest(20).index.values
top_20_idx = top_20_ids - 1

movies_df = pd.read_csv(
    url + "u.item", sep="|", encoding="latin-1", header=None,
    names=["item_id", "title"] + [f"c{i}" for i in range(22)],
    usecols=[0, 1]
)
titles = movies_df.set_index("item_id")["title"].to_dict()
labels = [titles.get(i, f"Movie {i}")[:25] for i in top_20_ids]

sim_subset = item_sim_full[np.ix_(top_20_idx, top_20_idx)]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(sim_subset, xticklabels=labels, yticklabels=labels,
            cmap="YlOrRd", vmin=0, vmax=1, ax=ax, square=True)
ax.set_title("Item-Item Cosine Similarity (Top-20 Most-Rated Movies)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("img/item_similarity_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

**Reflection — § 3 Item-Based CF**

Item-CF achieves ~68% HR@10 — a meaningful gain over user-CF (~60%). The improvement comes from stability: popular movies have hundreds of ratings, so their item-similarity vectors are dense and reliable. But item-CF still hits a wall at 68%, and the reason is fundamental rather than tunable.

The wall is sparsity in the opposite direction. For a user with only 15 ratings, item-CF can only recommend items similar to those 15. The 1,667 unrated movies might contain the user's all-time favourite — but if none of their rated films are similar to it, the model scores it zero. You can't find similar items when the user's history is too short. That is the sparsity problem, and no choice of $k$ fixes it. The fix requires a different representation: latent factors, which Ch.3 introduces.

## Summary

**What this chapter unlocked:**

| # | Constraint | Target | Ch.2 Result |
|---|-----------|--------|-------------|
| 1 | ACCURACY | >85% HR@10 | **~68%** (+26 pp from Ch.1) |
| 2 | COLD START | New users/items | Fails — no ratings = no similar users |
| 3 | SCALABILITY | 1M+ ratings | Item-CF: precomputed similarity (feasible); User-CF: O(m²) at scale |
| 4 | DIVERSITY | Not just popular | Better — neighbours have diverse taste |
| 5 | EXPLAINABILITY | "Because you liked X" | "Users like you also watched..." |

**Key takeaways:**
- User-based CF is intuitive but scales poorly ($O(m^2)$ similarity matrix)
- Item-based CF is more stable and precomputable; item popularity gives denser similarity vectors
- Mean-centering removes generous-vs-harsh rating bias before computing cosine similarity
- The 68% ceiling is a sparsity wall: users with thin history have too few anchors for neighbourhood prediction

**Checkpoint:** Item-based CF reaches 68% HR@10 on MovieLens 100k. FlixAI has its first personalised model — different users now receive different top-10 lists. The 17-point gap to 85% comes from sparse rating coverage that neighbourhood methods cannot bridge.

**Forward:** Ch.3 introduces matrix factorization — instead of storing a full $n \times n$ similarity matrix, MF compresses the entire ratings signal into two small dense matrices. Even users who never rated the same film can share a latent taste profile. HR@10 climbs to ~78%.

## Exercises

**Exercise 1 — Pearson vs Cosine**
Implement Pearson correlation for user-based CF (center each user's ratings by their mean before computing cosine similarity). Compare HR@10 against raw cosine.

**Exercise 2 — Minimum Overlap Threshold**
Modify item-based CF to require at least `min_overlap=5` co-rated items before computing similarity. How does this affect HR@10?

**Exercise 3 — Hybrid: Popularity + CF**
For users with fewer than 20 ratings, fall back to the popularity baseline. For users with ≥20 ratings, use item-based CF. Compare HR@10 against pure item-CF.

In [ ]:
# ── Exercise 1 scaffold — Pearson vs Cosine ───────────────────────────────
# TODO: Implement Pearson correlation for user-based CF
# Hint: Center each user's ratings by their mean before computing cosine

# R_centered = ... (subtract user means from non-zero entries)
# pearson_sim = cosine_similarity(R_centered)
# ... run user-based CF with pearson_sim, evaluate HR@10

pass

In [ ]:
# ── Exercise 2 scaffold — Minimum Overlap Threshold ──────────────────────
# TODO: Modify item similarity to require min_overlap co-rated items

# def item_sim_with_overlap(R_sparse, min_overlap=5):
#     """Set similarity to 0 for item pairs with fewer than min_overlap co-ratings."""
#     ...

pass

In [ ]:
# ── Exercise 3 scaffold — Hybrid Popularity + CF ─────────────────────────
# TODO: Use popularity for users with <20 ratings, item-CF for users with ≥20

# user_rating_counts = train.groupby("user_id").size()
# for user_id in test["user_id"].unique():
#     if user_rating_counts.get(user_id, 0) < 20:
#         recs = popularity_recs
#     else:
#         recs = item_cf_recs
# ... evaluate HR@10

pass